In [1]:
import matplotlib.pyplot as plt
from matplotlib import font_manager
# 设置支持中文的字体
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei']  
plt.rcParams['axes.unicode_minus'] = False 

In [4]:
import pandas as pd
import numpy as np

# 读取数据
df = pd.read_csv(r'/home/mw/input/data_total74037403/行业角度数据集/全国企业创新调查-主表.csv')

df['企业类型'] = df['企业规模'].apply(lambda x: '小微企业' if x in ['小型企业', '微型企业'] else '头部企业' if x == '大型企业' else '中型企业')
#df = df[df['企业类型'] != '中型企业']  # 移除中型企业

# 按年份和企业类型聚合
grouped = df.groupby(['年份', '企业类型']).agg({
    '企业数': 'sum',
    '创新费用支出合计(亿元)': 'sum',
    '开展创新合作企业数': 'sum',
    '采取了知识产权保护或相关措施的企业数': 'sum'
}).reset_index()


def gini_coefficient(x):
    x = np.sort(x)
    n = len(x)
    cumx = np.cumsum(x, dtype=float)
    return (n + 1 - 2 * np.sum(cumx) / cumx[-1]) / n

# 生成全体企业资源分配列表
def generate_resource_array(grouped_df, resource_col):
    resources = []
    for _, row in grouped_df.iterrows():
        num_enterprises = row['企业数']
        total_resource = row[resource_col]
        if num_enterprises > 0:
            avg_resource = total_resource / num_enterprises
            resources.extend([avg_resource] * num_enterprises)
    return np.array(resources)

# 按年份计算基尼系数
years = sorted(df['年份'].unique())
gini_results = []

for year in years:
    year_data = grouped[grouped['年份'] == year]
    resource_array = generate_resource_array(year_data, '创新费用支出合计(亿元)')
    if len(resource_array) > 0:
        gini = gini_coefficient(resource_array)
        gini_results.append({'年份': year, '基尼系数': gini})
    else:
        gini_results.append({'年份': year, '基尼系数': None})

gini_df = pd.DataFrame(gini_results)



In [6]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle

plt.figure(figsize=(10, 6))

# 绘制折线
plt.plot(gini_df['年份'], gini_df['基尼系数'], marker='o', linestyle='-', label='基尼系数', zorder=3)

# 渐变色设置
cmap = LinearSegmentedColormap.from_list('bar_grad', ['#fff0f6', '#ffd6e0', '#4F77CE'])
bar_width = 0.5

# 绘制渐变柱形
ax = plt.gca()
for xi, yi in zip(gini_df['年份'], gini_df['基尼系数']):
    for j in range(100):
        y0 = yi * j / 100
        y1 = yi * (j + 1) / 100
        color = cmap(j / 100)
        rect = Rectangle((xi - bar_width/2, y0), bar_width, y1 - y0, color=color, zorder=2, linewidth=0)
        ax.add_patch(rect)

# 在柱形和折线上方显示数据
for x, y in zip(gini_df['年份'], gini_df['基尼系数']):
    plt.text(x, y, f'{y:.3f}', ha='center', va='bottom', fontsize=10, color="#59ACDFEC")

plt.axhline(0.4, color='red', linestyle='--', label='警戒线（0.4）', zorder=4)
plt.title('创新费用支出的基尼系数变化（2016-2023）')
plt.xlabel('年份')
plt.ylabel('基尼系数')
plt.ylim(0.2, 0.8)  # y轴从0.2开始
plt.legend()
plt.show()

<Figure size 720x432 with 1 Axes>

In [8]:
import pandas as pd
import numpy as np

# 读取数据
df = pd.read_csv(r'/home/mw/input/data_total74037403/行业角度数据集/全国企业创新调查-主表.csv')

# 合并小型和微型企业
df['企业类型'] = df['企业规模'].apply(lambda x: '小微企业' if x in ['小型企业', '微型企业'] else '头部企业' if x == '大型企业' else '中型企业')

# 按年份和企业类型聚合
grouped = df.groupby(['年份', '企业类型']).agg({
    '企业数': 'sum',
    '创新费用支出合计(亿元)': 'sum',
    '开展创新合作企业数': 'sum',
    '采取了知识产权保护或相关措施的企业数': 'sum'
}).reset_index()

def gini_coefficient(x):
    x = np.sort(x)
    n = len(x)
    cumx = np.cumsum(x, dtype=float)
    return (n + 1 - 2 * np.sum(cumx) / cumx[-1]) / n

def generate_resource_array(grouped_df, resource_col):
    resources = []
    for _, row in grouped_df.iterrows():
        num_enterprises = row['企业数']
        total_resource = row[resource_col]
        if num_enterprises > 0:
            avg_resource = total_resource / num_enterprises
            resources.extend([avg_resource] * num_enterprises)
    return np.array(resources)

years = sorted(df['年份'].unique())
gini_results_coop = []
gini_results_ipr = []

for year in years:
    year_data = grouped[grouped['年份'] == year]
    # 创新合作
    arr_coop = generate_resource_array(year_data, '开展创新合作企业数')
    if len(arr_coop) > 0:
        gini_coop = gini_coefficient(arr_coop)
    else:
        gini_coop = None
    gini_results_coop.append({'年份': year, '基尼系数': gini_coop})
    # 知识产权保护
    arr_ipr = generate_resource_array(year_data, '采取了知识产权保护或相关措施的企业数')
    if len(arr_ipr) > 0:
        gini_ipr = gini_coefficient(arr_ipr)
    else:
        gini_ipr = None
    gini_results_ipr.append({'年份': year, '基尼系数': gini_ipr})

gini_df_coop = pd.DataFrame(gini_results_coop)
gini_df_ipr = pd.DataFrame(gini_results_ipr)

In [9]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(12, 6))

# 渐变粉色背景
ax = plt.gca()
ylim = (0, 0.8)  # y轴范围
from matplotlib.colors import LinearSegmentedColormap

cmap = LinearSegmentedColormap.from_list('pink_grad', ['#fff0f6', '#ffd6e0', "#f8c1ca"])
for i in range(100):
    plt.axhspan(ylim[0] + (ylim[1]-ylim[0])/100*i, ylim[0] + (ylim[1]-ylim[0])/100*(i+1),
                color=cmap(i/100), zorder=0)

# 绘制两条基尼系数曲线，分别用不同线型
plt.plot(
    gini_df_coop['年份'], gini_df_coop['基尼系数'],
    marker='o', linestyle='-', color="#2ca070",
    label='创新合作', zorder=2
)
plt.plot(
    gini_df_ipr['年份'], gini_df_ipr['基尼系数'],
    marker='o', linestyle='--', color="#4f77d6",
    label='知识产权保护', zorder=2
)

plt.axhline(0.4, color='gray', linestyle='--', label='警戒线（0.4）', zorder=3)
plt.title('创新资源分配的基尼系数变化（2016-2023）')
plt.xlabel('年份')
plt.ylabel('基尼系数')
plt.ylim(ylim)
plt.legend()
plt.grid(alpha=0.18, linewidth=1.2)
plt.show()

<Figure size 864x432 with 1 Axes>

In [10]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle

plt.figure(figsize=(12, 7))

# 渐变柱形（创新费用支出）
cmap = LinearSegmentedColormap.from_list('bar_grad', ['#fff0f6', '#ffd6e0', '#4F77CE'])
bar_width = 0.5
ax = plt.gca()
for xi, yi in zip(gini_df['年份'], gini_df['基尼系数']):
    for j in range(100):
        y0 = yi * j / 100
        y1 = yi * (j + 1) / 100
        color = cmap(j / 100)
        rect = Rectangle((xi - bar_width/2, y0), bar_width, y1 - y0, color=color, zorder=1, linewidth=0)
        ax.add_patch(rect)

# 折线（创新费用支出）
plt.plot(gini_df['年份'], gini_df['基尼系数'], marker='o', linestyle='-.', color="#4F77CE", label='创新费用支出', zorder=3)
for x, y in zip(gini_df['年份'], gini_df['基尼系数']):
    plt.text(x, y, f'{y:.3f}', ha='center', va='bottom', fontsize=10, color="#4F77CE")

# 折线（创新合作）
plt.plot(
    gini_df_coop['年份'], gini_df_coop['基尼系数'],
    marker='s', linestyle='-', color="#2ca07081",
    label='创新合作', zorder=4
)
for x, y in zip(gini_df_coop['年份'], gini_df_coop['基尼系数']):
    plt.text(x, y, f'{y:.3f}', ha='center', va='bottom', fontsize=10, color="#2ca070")

# 折线（知识产权保护）
plt.plot(
    gini_df_ipr['年份'], gini_df_ipr['基尼系数'],
    marker='^', linestyle='--', color="#eba272",
    label='知识产权保护', zorder=4
)
for x, y in zip(gini_df_ipr['年份'], gini_df_ipr['基尼系数']):
    plt.text(x, y, f'{y:.3f}', ha='center', va='bottom', fontsize=10, color="#eba272")

plt.axhline(0.4, color='red', linestyle=':', label='警戒线（0.4）', zorder=5)
plt.xlabel('年份')
plt.ylabel('基尼系数')
plt.title('创新资源分配的基尼系数变化（2016-2023）')
plt.ylim(0, 0.7)
plt.legend(loc='lower right', bbox_to_anchor=(1.0, 0.15))
plt.grid(alpha=0.18, linewidth=1.2)
plt.tight_layout()
plt.show()

<Figure size 864x504 with 1 Axes>